# FishWeight — execução em GPU (Kaggle, Colab ou PC com GPU)

Este notebook roda a parte pesada do projeto (treino de redes) fora do notebook pessoal.
O código, os testes e os relatórios continuam sendo escritos no repositório; aqui só se executa.

**Antes de rodar**

1. Ative a GPU. Colab: *Ambiente de execução → Alterar tipo → GPU*. Kaggle: *Settings → Accelerator → GPU* e *Internet → On*.
2. Confirme que o branch abaixo está no GitHub (`git push`). O notebook clona o repositório; o que não foi enviado não chega aqui.

Todas as etapas podem ser interrompidas e retomadas: o que já foi gravado em `results/` é pulado.

In [ ]:
REPO = "https://github.com/mpgxc/estimate_body_fish_weight.git"
BRANCH = "fase-1a-base-e-pisos"  # troque pelo branch que você quer executar

import os
if not os.path.isdir("estimate_body_fish_weight"):
    !git clone --depth 1 -b {BRANCH} {REPO}
%cd estimate_body_fish_weight
!pip install -q -e . pytest

## 1. Conferir o ambiente

O dispositivo precisa ser `cuda`. Se aparecer `cpu`, a GPU não está ativa.

In [ ]:
import torch
from fishweight.torch_utils import pick_device

print("PyTorch", torch.__version__, "| dispositivo:", pick_device())
if pick_device() == "cuda":
    print(torch.cuda.get_device_name(0))

## 2. Testes

A suíte inteira, incluindo os testes que treinam redes pequenas. Se algo falhar aqui, não siga adiante.

In [ ]:
!python -m pytest -q

## 3. Segmentação: 21 U-Nets, uma por dobra

Grava as máscaras previstas em `results/segmentation/` e as medidas em `data/features/predita_*.csv`.

In [ ]:
!python -m experiments.segmentation

## 4. Pisos (E0 e E1) com máscara manual e prevista

Rodam em segundos, com ou sem GPU.

In [ ]:
!python -m experiments.e0_allometric
!python -m experiments.e1_regionprops
!python -m experiments.check_scale

## 5. E2 — regressão direta da foto

Primeiro grava os vetores das quatro redes congeladas (rápido) e roda os regressores simples sobre eles.
Depois faz o ajuste fino de ResNet50, EfficientNet-B3 e ConvNeXt-Tiny, com o peso em gramas e em log.

Por padrão o ajuste fino roda só em P1 e P3 (6 dobras). Para incluir P2 (mais 15 dobras), acrescente `--protocols p1 p2 p3`.

In [ ]:
!python -m experiments.e2_direct

## 6. E3 — multitarefa

Parte do melhor ajuste fino de E2 em P3 e acrescenta saídas auxiliares de espécie e de medidas.

In [ ]:
!python -m experiments.e3_multitask

## 7. E4 e E5 — híbrido

Junta o vetor da foto com os atributos da máscara: em regressores simples (rede congelada) e por fusão dentro da rede (ajuste fino).

In [ ]:
!python -m experiments.e4_hybrid

## 8. F1 e F2 — dataset público FDWE

Baixa o FDWE (2,4 GB, licença CC BY 4.0) para `data/external/fdwe/`, recorta cada peixe pela caixa anotada e roda:

- **F2**: treina num conjunto e testa no outro, sem ajuste, nos dois sentidos;
- **F1**: pré-treina no FDWE e faz o ajuste fino na coleta.

Usa a melhor rede de E2. Só entram peixes com até 30% de oclusão; mude com `--max-occlusion`.

In [ ]:
!python -m experiments.f_fdwe

## 9. Tabela de comparação

In [ ]:
!python -m experiments.report
!head -40 docs/experiments/comparacao.md

## 10. Empacotar os resultados

Gera `resultados_fishweight.zip`. Baixe o arquivo e, no seu computador, descompacte na raiz do repositório:

```bash
unzip -o resultados_fishweight.zip -d /caminho/para/estimate_body_fish_weight
```

In [ ]:
import pathlib, zipfile

destino = pathlib.Path("/kaggle/working" if os.path.isdir("/kaggle/working") else "..") / "resultados_fishweight.zip"
pastas = ["results", "data/features", "docs/experiments/comparacao.md"]
with zipfile.ZipFile(destino, "w", zipfile.ZIP_DEFLATED) as pacote:
    for pasta in pastas:
        caminho = pathlib.Path(pasta)
        for arquivo in ([caminho] if caminho.is_file() else sorted(caminho.rglob("*"))):
            if arquivo.is_file():
                pacote.write(arquivo)
print(destino.resolve(), f"{destino.stat().st_size / 1e6:.1f} MB")